# 循環神經網絡

在上一個模組中，我們介紹了文本的豐富語義表示。我們一直使用的架構捕捉了句子中單詞的綜合意義，但它沒有考慮單詞的<strong>順序</strong>，因為之後對詞嵌入所進行的聚合操作會移除原始文本中的這種信息。由於這些模型無法表示單詞的順序，它們無法解決更複雜或模糊的任務，例如文本生成或問答。

為了捕捉文本序列的意義，我們將使用一種稱為<strong>循環神經網絡</strong>（RNN）的神經網絡架構。使用RNN時，我們會一次將句子中的一個標記傳遞到網絡中，網絡產生一些<strong>狀態</strong>，然後我們再將這個狀態與下一個標記一起傳入網絡。

![顯示循環神經網絡生成範例的圖片。](../../../../../translated_images/zh-MO/rnn.27f5c29c53d727b5.webp)

給定輸入標記序列 $X_0,\dots,X_n$，RNN 創建一個神經網絡模塊序列，並使用反向傳播對該序列進行端到端的訓練。每個網絡模塊以一對 $(X_i,S_i)$ 作為輸入，並產生 $S_{i+1}$ 作為結果。最終狀態 $S_n$ 或輸出 $Y_n$ 輸入到線性分類器以產生結果。所有網絡模塊共享相同的權重，並通過一次反向傳播過程端到端進行訓練。

> 上圖展示了循環神經網絡的展開形式（左側），以及更緊湊的循環表示（右側）。重要的是要明白，所有RNN單元具有相同的<strong>可共享權重</strong>。

由於狀態向量 $S_0,\dots,S_n$ 在網絡中傳遞，RNN 能夠學習單詞之間的順序依賴關係。例如，當序列中出現單詞<em>not</em>時，它可以學會否定狀態向量中的某些元素。

在結構中，每個RNN單元包含兩個權重矩陣：$W_H$ 和 $W_I$，以及偏置 $b$。在每個RNN步驟中，給定輸入 $X_i$ 和輸入狀態 $S_i$，輸出狀態計算為 $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$，其中 $f$ 是激活函數（通常是 $\tanh$）。

> 對於文本生成（我們將在下一單元介紹）或機器翻譯等問題，我們還希望在每個RNN步驟獲得一些輸出值。在這種情況下，還有另一個矩陣 $W_O$，輸出計算為 $Y_i=f(W_O\times S_i+b_O)$。

讓我們看看循環神經網絡如何幫助我們分類新聞數據集。

> 對於沙箱環境，我們需要運行以下程式碼格，以確保所需的庫已安裝並且數據已預抓取。如果您在本地執行，可以跳過以下程式碼格。


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

在訓練大型模型時，GPU 記憶體配置可能會成為問題。我們也可能需要嘗試不同的迷你批次大小，以便資料能夠適合我們的 GPU 記憶體，同時訓練速度足夠快。如果你是在自己的 GPU 機器上運行這段程式碼，可以嘗試調整迷你批次大小來加快訓練速度。

> <strong>注意</strong>：某些版本的 NVidia 驅動程式已知在訓練模型後不會釋放記憶體。我們在這個筆記本中運行了多個範例，這可能會在某些設置中導致記憶體耗盡，特別是如果你正在同一本筆記本中做自己的實驗。如果你在開始訓練模型時遇到一些奇怪的錯誤，建議你重新啟動筆記本核心。


In [3]:
batch_size = 16
embed_size = 64

## 簡易 RNN 分類器

對於簡單的 RNN，每個循環單元都是一個簡單的線性網絡，它接收一個輸入向量和狀態向量，並產生一個新的狀態向量。在 Keras 中，這可以用 `SimpleRNN` 層表示。

雖然我們可以直接將 one-hot 編碼的代幣傳入 RNN 層，但由於其高維度，這並不是一個好主意。因此，我們將使用一層嵌入層來降低詞向量的維度，接著是 RNN 層，最後是一個 `Dense` 分類器。

> <strong>注意</strong>：在維度不是那麼高的情況下，例如使用字符級分詞時，直接將 one-hot 編碼的代幣傳入 RNN 單元可能是合理的。


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **注意：** 這裡我們使用未訓練的嵌入層以簡化，但為了獲得更好的效果，我們可以使用如前一單元所描述的 Word2Vec 預訓練嵌入層。將此程式碼調整為使用預訓練嵌入層會是個很好的練習。

現在讓我們訓練 RNN。RNN 一般來說相當難以訓練，因為一旦 RNN 單元沿著序列長度展開，反向傳播所涉及的層數會非常多。因此我們需要選擇較小的學習率，並在較大的數據集上訓練網絡以產生良好結果。這可能會花費相當長的時間，所以建議使用 GPU。

為了加快速度，我們將僅在新聞標題上訓練 RNN 模型，忽略描述。你可以嘗試加上描述一起訓練，看看是否能讓模型成功訓練。


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> <strong>注意</strong> 這裡的準確度可能較低，因為我們只在新聞標題上進行訓練。


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


現在我們使用遮罩，可以在整個標題和描述數據集上訓練模型。 

> <strong>注意</strong>：你有沒有注意到我們使用的是在新聞標題上訓練的向量化器，而不是整篇文章的內容？這可能導致部分字元被忽略，因此最好重新訓練向量化器。但這影響可能非常小，為簡便起見，我們將繼續使用之前的預訓練向量化器。 


## LSTM：長短期記憶

RNN 的主要問題之一是<strong>梯度消失</strong>。RNN 可以非常長，在反向傳播時可能難以將梯度一直傳回網絡的第一層。當這種情況發生時，網絡就無法學習遠距離的符號之間的關係。避免這個問題的一種方法是通過使用<strong>閘門</strong>來引入<strong>顯式狀態管理</strong>。引入閘門的兩種最常見架構是<strong>長短期記憶</strong>（LSTM）和<strong>門控遞歸單元</strong>（GRU）。我們在這裡會介紹 LSTM。

![顯示長短期記憶單元範例的圖片](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM 網絡的組織方式與 RNN 類似，但有兩個狀態會從一層傳到另一層：真實狀態 $c$ 和隱藏向量 $h$。在每個單元中，隱藏向量 $h_{t-1}$ 會與輸入 $x_t$ 結合，一同通過<strong>閘門</strong>控制狀態 $c_t$ 和輸出 $h_t$ 的變化。每個閘門的激活函數都是 sigmoid（輸出範圍在 $[0,1]$ 之間），乘以狀態向量時可視為位元遮罩。LSTM 有以下閘門（由左至右如上圖所示）：
* <strong>遺忘閘門</strong>，決定向量 $c_{t-1}$ 中哪些部分需要遺忘，哪些需要保留。
* <strong>輸入閘門</strong>，決定有多少來自輸入向量和先前隱藏向量的信息應納入狀態向量。
* <strong>輸出閘門</strong>，根據新的狀態向量決定使用哪些部分來產生新的隱藏向量 $h_t$。

狀態 $c$ 的組成部分可視為可開關的標誌。例如，當序列中出現名字 *Alice* 時，我們推測它指的是女性，於是在狀態中設置一個標誌，表示句子中有一個女性名詞。接著當我們遇到詞語 *and Tom* 時，我們會設置另一個標誌，表示句子中有複數名詞。透過操作狀態，我們就能追蹤句子的文法屬性。

> <strong>注意</strong>：這是一個很棒的資源，幫助理解 LSTM 的內部結構：[Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) 作者 Christopher Olah。

雖然 LSTM 單元的內部結構看起來很複雜，但 Keras 將這個實現封裝在 `LSTM` 層中，因此在上面的範例中，我們唯一需要做的就是替換遞歸層：


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> <strong>注意</strong> 訓練 LSTM 亦相當緩慢，而在訓練初期你可能看不到太大準確度提升。你或需繼續訓練一段時間才能達到良好準確度。


## 雙向及多層 RNN

在我們迄今為止的例子中，循環網絡都是從序列的開始運行到結束。這對我們來說感覺自然，因為它遵循我們閱讀或聽語音時的相同方向。然而，對於需要隨機存取輸入序列的情境，讓循環運算在雙方向上執行會更合理。允許在兩個方向上進行運算的 RNN 稱為<strong>雙向</strong> RNN，它們可透過用特別的 `Bidirectonal` 層包裝循環層來建立。

> <strong>注意</strong>：`Bidirectional` 層會複製它內部的層兩次，並將其中一個複本的 `go_backwards` 屬性設為 `True`，使其沿序列朝相反方向運行。

循環網絡，無論是單向或雙向，都會捕捉序列中的模式，並將其存入狀態向量或作為輸出。和卷積網絡一樣，我們可以在第一層之後建立另一層循環層，以捕捉由第一層提取的較低層次模式組成的較高層次模式。這就引出了<strong>多層 RNN</strong>的概念，它由兩個或以上循環網絡組成，先前層的輸出被傳遞給下一層作為輸入。

![顯示多層長短期記憶 RNN 的圖片](../../../../../translated_images/zh-MO/multi-layer-lstm.dd975e29bb2a59fe.webp)

*圖片來自 Fernando López 的[這篇精彩文章](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3)。*

使用 Keras 建構這些網絡非常簡單，因為你只需在模型中添加更多循環層。對於除最後一層之外的所有層，我們需要指定 `return_sequences=True` 參數，因為我們需要該層返回所有中間狀態，而非僅返回循環運算的最終狀態。

現在我們來為分類問題建立一個雙層雙向 LSTM。

> <strong>注意</strong> 此程式碼執行起來同樣需要相當長的時間，但它能給我們迄今為止最高的準確率。因此，也許值得等待並觀察結果。


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN 用於其他任務

到目前為止，我們專注於使用 RNN 來分類文本序列。但它們能處理更多任務，例如文本生成和機器翻譯 —— 我們會在下一單元中探討這些任務。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
